# CSET485 Assignment 1: COMPAS Recidivism Analysis

This notebook implements the assignment for a roll number ending in **2**. The random seed is **182**, using the last three digits of 0182. It downloads ProPublica's `compas-scores-two-years.csv`, runs the analysis from top to bottom, and writes tables, test predictions, the sampled error subset, and the fairness chart to a `work/` folder.

The assignment calls the target `is_recidivist`. The supplied COMPAS file contains `two_year_recid`, a binary indicator for recidivism within two years. This notebook uses that field as the requested binary outcome. It uses the full supplied file and only removes a row if one of the selected analysis fields is missing. The assignment's selected columns have no missing values in this file.

Run from the repository root or its `outputs/` folder. The first run needs internet access. Required packages: pandas, numpy, scikit-learn, statsmodels, and matplotlib. Install them in the notebook environment if needed with `%pip install pandas numpy scikit-learn statsmodels matplotlib`.

## Setup, data, preprocessing, split, and model fit

Predictors: age, sex, race, juvenile felony/misdemeanor/other counts, prior counts, and charge degree. Count features use `log1p`; numeric features are standardized. Categorical fields are one-hot encoded. Transformations are fit on the training data only.

In [ ]:
import os, json, urllib.request, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
ROOT=Path.cwd()
if ROOT.name == 'outputs': ROOT=ROOT.parent
WORK=ROOT/'work'; OUT=WORK; WORK.mkdir(exist_ok=True)
URL='https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv'
DATA=WORK/'compas-scores-two-years.csv'
if not DATA.exists():
    urllib.request.urlretrieve(URL, DATA)
raw=pd.read_csv(DATA)

# The assignment's binary two-year recidivism outcome is represented by this column.
# Keep only analysis fields; no records are removed for outlier values.
num=['age','juv_fel_count','juv_misd_count','juv_other_count','priors_count']
cat=['sex','race','c_charge_degree']
target='two_year_recid'
decile='decile_score'
df=raw[num+cat+[target,decile]].copy()
for c in num+[target,decile]: df[c]=pd.to_numeric(df[c],errors='coerce')
df=df.dropna(subset=[target,decile]+num+cat).copy()
df[target]=df[target].astype(int)
df[decile]=df[decile].astype(float)
df['age_band']=np.where(df.age<25,'Under 25','25 and over')
# Use log1p for nonnegative count predictors to reduce skew while retaining zeros.
count_cols=['juv_fel_count','juv_misd_count','juv_other_count','priors_count']
for c in count_cols:
    if (df[c]<0).any(): raise ValueError(f'Negative count in {c}')

seed=182
X=df[num+cat]
y=df[target]
ydec=df[decile]
idx_train,idx_test=train_test_split(np.arange(len(df)),test_size=.30,random_state=seed,stratify=y)
X_train,X_test=X.iloc[idx_train],X.iloc[idx_test]
y_train,y_test=y.iloc[idx_train],y.iloc[idx_test]
dec_train,dec_test=ydec.iloc[idx_train],ydec.iloc[idx_test]

# Log-count features are created before scaling; all learned transforms fit on training data only.
def make_features(frame):
    z=frame.copy()
    for c in count_cols: z[c]=np.log1p(z[c])
    return z
num_pipe=Pipeline([('log_counts',FunctionTransformer(make_features,validate=False)),('scale',StandardScaler())])
pre=ColumnTransformer([('num',num_pipe,num),('cat',OneHotEncoder(drop='first',handle_unknown='ignore'),cat)])
lin=Pipeline([('prep',pre),('model',LinearRegression())])
log=Pipeline([('prep',pre),('model',LogisticRegression(C=1.0,max_iter=3000,solver='lbfgs'))])
lin.fit(X_train,dec_train)
log.fit(X_train,y_train)
linear_pred=lin.predict(X_test)
linear_median=float(np.median(linear_pred))
class_lin=(linear_pred>=linear_median).astype(int)
prob=log.predict_proba(X_test)[:,1]
class_log=(prob>=.5).astype(int)
cm=confusion_matrix(class_lin,class_log,labels=[0,1])
mismatch=(class_lin!=class_log)



## Part 1: Model comparison

Model A predicts the continuous COMPAS decile score and thresholds at the median predicted test value. Model B predicts the two-year binary outcome and thresholds at 0.5. Confusion-matrix rows are linear classifications and columns are logistic classifications.

In [ ]:
print('Test rows:', len(y_test))
print('Median linear prediction:', round(linear_median, 4))
print('Confusion matrix, rows=linear low/high, columns=logistic low/high:')
print(cm)
print('Mismatches:', int(mismatch.sum()), f'({mismatch.mean()*100:.2f}%)')
print('Logistic accuracy at 0.5:', round(accuracy_score(y_test, class_log)*100, 2), '%')

## Part 2.1: Subgroup effects

Compare the full sample with African-American and Caucasian subgroup models. Coefficients are on the log-odds scale. The interaction fit tests whether the juvenile other-offense coefficient differs between the two groups.

In [ ]:
# Inference models use the full complete-case analytic sample and log-count predictors.
df['log_juv_fel_count']=np.log1p(df.juv_fel_count)
df['log_juv_misd_count']=np.log1p(df.juv_misd_count)
df['log_juv_other_count']=np.log1p(df.juv_other_count)
df['log_priors_count']=np.log1p(df.priors_count)
base_terms=['age','log_juv_fel_count','log_juv_misd_count','log_juv_other_count','log_priors_count','C(sex)','C(race)','C(c_charge_degree)']
formula=target+' ~ '+' + '.join(base_terms)
fit=smf.glm(formula,data=df,family=sm.families.Binomial()).fit()
# Race subgroup comparison: African-American and Caucasian groups; full model is all races.
sub_formula=target+' ~ age + log_juv_fel_count + log_juv_misd_count + log_juv_other_count + log_priors_count + C(sex) + C(c_charge_degree)'
subfits={}
for race in ['African-American','Caucasian']:
    subfits[race]=smf.glm(sub_formula,data=df[df.race==race],family=sm.families.Binomial()).fit()
# Formal test of whether prior-count effect differs by race in Black/White comparison.
bw=df[df.race.isin(['African-American','Caucasian'])].copy()
bw['is_black']=(bw.race=='African-American').astype(int)
int_fit=smf.glm(target+' ~ age + log_juv_fel_count + log_juv_misd_count + log_juv_other_count + log_priors_count*is_black + C(sex) + C(c_charge_degree)',data=bw,family=sm.families.Binomial()).fit()
race_other_fit=smf.glm(target+' ~ age + log_juv_fel_count + log_juv_misd_count + log_juv_other_count*C(race) + log_priors_count + C(sex) + C(c_charge_degree)',data=bw,family=sm.families.Binomial()).fit()



## Part 2.2: Omitted-variable test

Age is removed from the full model. Percentage change is calculated as `(reduced - full) / abs(full) * 100`; this makes a percentage large when the original coefficient is close to zero.

In [ ]:
# Omitted-variable test: age is removed from a model where it is a plausible confounder.
reduced_formula=target+' ~ '+' + '.join(t for t in base_terms if t!='age')
reduced=smf.glm(reduced_formula,data=df,family=sm.families.Binomial()).fit()
compare_terms=['log_priors_count','log_juv_fel_count','log_juv_misd_count','log_juv_other_count','C(sex)[T.Male]','C(c_charge_degree)[T.M]']
ovb=[]
for term in compare_terms:
    a=float(fit.params.get(term,np.nan)); b=float(reduced.params.get(term,np.nan))
    ovb.append({'term':term,'full':a,'reduced':b,'pct_change':(b-a)/abs(a)*100 if a else np.nan})



## Part 2.3: Confident errors

Select test cases with predicted probability at least 0.75 and actual outcome 0, or probability at most 0.25 and actual outcome 1. If more than 50 exist, sample 50 with seed 182.

In [ ]:
# Adversarial test cases: high-confidence false positives/negatives.
wrong=((prob>=.75)&(y_test.to_numpy()==0))|((prob<=.25)&(y_test.to_numpy()==1))
wrong_idx=np.where(wrong)[0]
wrong_all=X_test.iloc[wrong_idx].copy()
wrong_all['actual']=y_test.iloc[wrong_idx].to_numpy()
wrong_all['probability']=prob[wrong_idx]
if len(wrong_all)>50:
    adv=wrong_all.sample(50,random_state=seed)
else: adv=wrong_all.copy()
adv_summary={
 'test_n':int(len(X_test)), 'wrong_n':int(len(wrong_all)), 'wrong_pct_test':float(len(wrong_all)/len(X_test)*100),
 'subset_n':int(len(adv)), 'subset_age_mean':float(adv.age.mean()), 'subset_age_median':float(adv.age.median()),
 'test_age_mean':float(X_test.age.mean()), 'test_age_median':float(X_test.age.median()),
 'subset_sex_pct':(adv.sex.value_counts(normalize=True)*100).round(2).to_dict(),
 'test_sex_pct':(X_test.sex.value_counts(normalize=True)*100).round(2).to_dict(),
 'subset_race_pct':(adv.race.value_counts(normalize=True)*100).round(2).to_dict(),
 'test_race_pct':(X_test.race.value_counts(normalize=True)*100).round(2).to_dict(),
 'subset_priors_ge5_pct':float((adv.priors_count>=5).mean()*100),
 'test_priors_ge5_pct':float((X_test.priors_count>=5).mean()*100),
 'subset_under25_pct':float((adv.age<25).mean()*100), 'test_under25_pct':float((X_test.age<25).mean()*100),
 'subset_felony_pct':float((adv.c_charge_degree=='F').mean()*100), 'test_felony_pct':float((X_test.c_charge_degree=='F').mean()*100)
}
adv.to_csv(OUT/'adversarial_subset.csv',index=True,index_label='test_row_index')

test_audit=df.iloc[idx_test][['age','sex','race','priors_count','c_charge_degree',target,decile]].copy()
test_audit['linear_decile_prediction']=linear_pred
test_audit['logistic_probability']=prob
test_audit['linear_high_risk']=(class_lin==1)
test_audit['logistic_high_risk_at_0_5']=(class_log==1)
test_audit.to_csv(OUT/'test_set_predictions.csv',index_label='dataset_row_index')



## Part 2.4: Fairness metrics and chart

Group A is African-American and Group B is Caucasian. Other race categories are outside the two-group comparison. Ratios follow the formulas in the assignment. At threshold 0.95, no cases are predicted positive, so DP, EO, and PP are undefined.

In [ ]:
# Fairness ratios for African-American (A) vs Caucasian (B); other races are outside this two-group comparison.
mask_test=df.iloc[idx_test].race.isin(['African-American','Caucasian']).to_numpy()
group_test=df.iloc[idx_test].race.to_numpy()
ytest=y_test.to_numpy()
fair=[]
def ratio(a,b):
    return float(a/b) if b!=0 else (float('nan') if a==0 else float('inf'))
for t in np.round(np.arange(.05,1.0,.1),2):
    pred=(prob>=t).astype(int)
    rates={}
    for g in ['African-American','Caucasian']:
        m=group_test==g
        pos=m&(pred==1); neg=m&(pred==0)
        actual1=m&(ytest==1); actual0=m&(ytest==0)
        rates[g]={
          'dp':float(pred[m].mean()),
          'tpr':float((pred[actual1]==1).mean()) if actual1.sum() else np.nan,
          'fpr':float((pred[actual0]==1).mean()) if actual0.sum() else np.nan,
          'ppv':float(ytest[pos].mean()) if pos.sum() else np.nan
        }
    a,b=rates['African-American'],rates['Caucasian']
    fair.append({'threshold':float(t),'DP':ratio(a['dp'],b['dp']), 'EO':min(ratio(a['tpr'],b['tpr']),ratio(a['fpr'],b['fpr'])), 'PP':ratio(a['ppv'],b['ppv']),
                 'A_selection':a['dp'],'B_selection':b['dp'],'A_TPR':a['tpr'],'B_TPR':b['tpr'],'A_FPR':a['fpr'],'B_FPR':b['fpr'],'A_PPV':a['ppv'],'B_PPV':b['ppv']})
fairdf=pd.DataFrame(fair)
fairdf.to_csv(OUT/'fairness_metrics.csv',index=False)



## Save results and review tables

The complete results dictionary is saved as JSON. CSVs include the fairness sweep, test-set predictions, and sampled adversarial rows. The chart is saved as a PNG. The code prints the report-ready coefficients and all fairness ratios.

In [ ]:
# Accuracy of logistic model at 0.5; compare threshold sweep values.
accuracy=accuracy_score(y_test,class_log)
# Pick a documented pair with the strongest DP improvement from the closest-to-1 DP threshold,
# among pairs where equalized-odds ratio falls.
valid=fairdf.replace([np.inf,-np.inf],np.nan).dropna(subset=['DP','EO'])
pairs=[]
for i in range(len(valid)):
 for j in range(len(valid)):
  first=valid.iloc[i]; second=valid.iloc[j]
  gain=abs(first.DP-1)-abs(second.DP-1)
  loss=first.EO-second.EO
  if gain>0 and loss>0:
   pairs.append((gain,loss,first,second))
pair=max(pairs,key=lambda q:q[0]) if pairs else None

# Plot required chart with assignment axes (0 to 1.5). List exact points outside range beside it.
fig=plt.figure(figsize=(10,5.6))
ax=fig.add_axes([.10,.16,.57,.76])
for row in fair:
    if np.isfinite(row['DP']) and np.isfinite(row['EO']) and row['DP']<=1.5 and row['EO']<=1.5:
        ax.scatter(row['DP'],row['EO'],s=48,color='#176b87')
        ax.annotate(f"{row['threshold']:.2f}",(row['DP'],row['EO']),xytext=(4,4),textcoords='offset points',fontsize=8)
ax.set(xlim=(0,1.5),ylim=(0,1.5),xlabel='Demographic parity ratio (African-American / Caucasian)',ylabel='Equalized odds ratio',title='Fairness trade-offs by decision threshold')
ax.grid(alpha=.25)
fig.text(.72,.88,'Thresholds outside chart range',fontsize=10,fontweight='bold')
outside=[r for r in fair if np.isfinite(r['DP']) and np.isfinite(r['EO']) and (r['DP']>1.5 or r['EO']>1.5)]
for j,row in enumerate(outside):
    fig.text(.72,.81-j*.07,f"t={row['threshold']:.2f}: DP {row['DP']:.2f}, EO {row['EO']:.2f}",fontsize=9)
fig.text(.72,.81-len(outside)*.07,'t=0.95: DP, EO, PP undefined',fontsize=9)
fig.text(.72,.15,'Exact values for all thresholds\nare listed in the table.',fontsize=8,color='#555555')
fig.savefig(OUT/'fairness_tradeoff.png',dpi=180); plt.close(fig)

# Build a compact results payload for the report writer.
def coef(fit,term):
 return {'coef':float(fit.params[term]),'ci_low':float(fit.conf_int().loc[term,0]),'ci_high':float(fit.conf_int().loc[term,1]),'p':float(fit.pvalues[term])}
key='log_priors_count'
subgroups={r:coef(m,key) for r,m in subfits.items()}
key_other='log_juv_other_count'
subgroups_other={r:coef(m,key_other) for r,m in subfits.items()}
results={
 'source_url':URL,'dataset_shape_raw':list(raw.shape),'n':int(len(df)),'target_counts':df[target].value_counts().sort_index().to_dict(),
 'missing_analysis_columns':raw[num+cat+[target,decile]].isna().sum().to_dict(),
 'seed':seed,'train_n':len(idx_train),'test_n':len(idx_test),'linear_median_pred':linear_median,
 'part1_confusion_rows_linear_cols_logistic':cm.tolist(),'part1_mismatches':int(mismatch.sum()),'part1_mismatch_pct':float(mismatch.mean()*100),
 'part1_logistic_accuracy_at_05':float(accuracy),'test_reconversion_pct':float(y_test.mean()*100),
 'subgroups':subgroups,'full_priors':coef(fit,key),'race_interaction_black_x_priors':coef(int_fit,'log_priors_count:is_black'),
 'subgroups_juvenile_other':subgroups_other,'full_juvenile_other':coef(fit,key_other),
 'race_interaction_white_x_juvenile_other':coef(race_other_fit,'log_juv_other_count:C(race)[T.Caucasian]'),
 'ovb':ovb,'ovb_max_abs_pct_term':max(ovb,key=lambda x:abs(x['pct_change']))['term'],
 'adversarial':adv_summary,'fairness':fair,
 'fairness_pair':None if pair is None else {'from_threshold':float(pair[2].threshold),'to_threshold':float(pair[3].threshold),'from_DP':float(pair[2].DP),'to_DP':float(pair[3].DP),'from_EO':float(pair[2].EO),'to_EO':float(pair[3].EO),'DP_error_gain_pct':float(pair[0]/abs(pair[2].DP-1)*100),'EO_loss_pct':float(pair[1]/abs(pair[2].EO)*100) if pair[2].EO else None},
 'race_sizes_test':{r:int((group_test==r).sum()) for r in ['African-American','Caucasian']},
 'threshold_accuracy':{str(float(t)):float(accuracy_score(y_test,(prob>=t).astype(int))) for t in np.round(np.arange(.05,1.0,.1),2)},
 'full_model_nobs':int(fit.nobs),'full_model_pseudo_r2':float(fit.pseudo_rsquared())
}
# Include coefficient table for all terms and group rates for threshold .5.
results['full_coefs']={term:coef(fit,term) for term in fit.params.index}
results['subgroup_counts']={r:int((df.race==r).sum()) for r in ['African-American','Caucasian']}
with open(OUT/'analysis_results.json','w') as f: json.dump(results,f,indent=2,allow_nan=True)
print(json.dumps(results,indent=2,allow_nan=True))
print('\nFairness table:\n',fairdf[['threshold','DP','EO','PP']].to_string(index=False))
print('\nOVB:\n',pd.DataFrame(ovb).to_string(index=False))


In [ ]:
print('Fairness table')
print(fairdf[['threshold','DP','EO','PP']].round(4).to_string(index=False))
print('Omitted-variable table')
print(pd.DataFrame(ovb).round(4).to_string(index=False))
print('Subgroup juvenile other-count coefficients')
print('Full:', coef(fit, 'log_juv_other_count'))
print('African-American:', coef(subfits['African-American'], 'log_juv_other_count'))
print('Caucasian:', coef(subfits['Caucasian'], 'log_juv_other_count'))
print('Race interaction:', coef(race_other_fit, 'log_juv_other_count:C(race)[T.Caucasian]'))
print('Files written to:', OUT)